# Improvement

1. Adding the GPU process
2. Loading data using polars
3. Adding tensorboard
4. Modifying the layer size
5. Adding early stopping
6. Change the relu into leaky_relu

In [ ]:
# Import required libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split

import pandas as pd
import numpy as np

import polars as pl

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from datetime import datetime
import time

In [ ]:
current_time = datetime.now().strftime("%Y%m%d%H%M%S")

progress_file = f'/group/pmc021/amunif/epi-thesis/workflow/04. Neural Network/05_optimize_{current_time}.txt'
dataset_path = "/group/pmc021/amunif/epi-thesis/dataset/"

In [ ]:
'''
User defined functions
'''
def load_large_csv(file_name, chunksize=20000):
    # Read the CSV file
    mylist = []

    for chunk in pd.read_csv(file_name, chunksize = chunksize):
        mylist.append(chunk)

    df = pd.concat(mylist, axis = 0)
    
    del mylist
    return df

def save_progress(file_name, message):
    with open(file_name, 'a+') as file:
        file.write(message + "\n")

## Loading the Data

In [ ]:
# Loading file using polars
X = pl.read_csv(f"{dataset_path}histone_features.csv")
y = pl.read_csv(f"{dataset_path}value_1_df.csv")

In [ ]:
# Convert to numpy
X_np = X.to_numpy()
y_np = y.to_numpy()

In [ ]:
# Split the data into training and testing
X_train_np, X_test_np, y_train_np, y_test_np = train_test_split(X_np, y_np, test_size=0.2, random_state=42)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
save_progress(progress_file, f'Using device: {device}')